In [ ]:
!pip install -q voyageai rank_bm25 transformers torch tqdm faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 17.0 MB/s eta 0:00:00


In [ ]:
import json
import pickle
import re
import os
import numpy as np
import torch
import faiss
import voyageai
from transformers import AutoModelForMaskedLM, AutoTokenizer
from tqdm import tqdm
from google.colab import drive

import openai
from google.colab import userdata

openai_client = openai.OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)

print("✓ openai_client initialized")

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/iam_rag_project/'
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'indexes/')
CHUNK_DIR  = os.path.join(DRIVE_BASE, 'data/processed/chunking/')

# Dense paths
DENSE_CHUNKS_PATH = os.path.join(OUTPUT_DIR, 'dense', 'combined_chunks.json')
DENSE_INDEX_PATH  = os.path.join(OUTPUT_DIR, 'dense', 'combined_faiss.index')

# Sparse paths
SPLADE_CHUNKS_PATH = os.path.join(OUTPUT_DIR, 'sparse', 'combined_chunks_splade.json')

# BM25 paths
BM25_INDEX_PATH  = os.path.join(OUTPUT_DIR, 'bm25', 'combined_bm25.pkl')
BM25_CHUNKS_PATH = os.path.join(OUTPUT_DIR, 'bm25', 'combined_chunks_mapped.json')

print("✓ Setup complete")
print(f"  DRIVE_BASE:  {DRIVE_BASE}")
print(f"  OUTPUT_DIR:  {OUTPUT_DIR}")

✓ openai_client initialized
Mounted at /content/drive
✓ Setup complete
  DRIVE_BASE:  /content/drive/MyDrive/iam_rag_project/
  OUTPUT_DIR:  /content/drive/MyDrive/iam_rag_project/indexes/


In [ ]:
#Shared utilities used by all three retrievers

def tokenize_for_bm25(text):
    """
    Tokenize text for BM25.
    Preserves IAM-specific syntax:
      - colon kept:  s3:GetObject stays as one token
      - hyphen kept: access-analyzer stays as one token
      - everything else lowercased and split on whitespace
    """
    text = text.lower()
    text = re.sub(r'[^\w\s:\-]', ' ', text)
    return text.split()


def format_results(chunks, scores, indices):
    """
    Standard result format used by all three retrievers.
    Returns list of dicts with text, metadata, score.

    Why this format:
      text     → for debugging, showing what was retrieved
      metadata → passed to GPT-3.5 during generation
                 contains action name, description,
                 resource_types, policy_document etc.
      score    → for comparing retriever performance
                 and re-ranking if needed
    """
    results = []
    for score, idx in zip(scores, indices):
        chunk = chunks[idx]
        results.append({
            'text':     chunk['text'],
            'metadata': chunk['metadata'],
            'score':    float(score)
        })
    return results


def print_results(results, query, method_name):
    """
    Pretty print retrieval results for inspection.
    """
    print(f"\n{'='*60}")
    print(f"Method: {method_name}")
    print(f"Query:  {query}")
    print(f"{'='*60}")

    for i, r in enumerate(results):
        metadata   = r['metadata']
        chunk_type = metadata['type']

        if chunk_type == 'iam_action':
            identifier = metadata.get('action', 'unknown')
            detail     = metadata.get('description', '')[:60]
        else:
            identifier = metadata.get('policy_name', 'unknown')
            detail     = f"{len(metadata.get('actions_used', []))} actions"

        print(f"\n  Rank {i+1}: [{chunk_type:<10}] {identifier}")
        print(f"          score={r['score']:.4f}")
        print(f"          {detail}...")


def weighted_rrf(ranked_lists, weights, k=60):
    """
    Weighted RRF — each retriever's contribution
    multiplied by its weight before summing.
    """
    from collections import defaultdict
    scores = defaultdict(float)

    for ranked_list, weight in zip(ranked_lists, weights):
        for rank, chunk_idx in enumerate(ranked_list):
            scores[chunk_idx] += weight * (1.0 / (k + rank + 1))

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

print("✓ Shared utilities defined")

✓ Shared utilities defined


In [ ]:
# Dense Retriever
#
# Logic:
#   INDEXING (done during embedding, not here):
#     Each chunk text was embedded into a 1024-dim vector
#     using voyage-code-3 with input_type="document"
#     All vectors stored in FAISS IndexFlatIP
#
#   RETRIEVAL (done here):
#     1. Embed query using voyage-code-3 input_type="query"
#        → IMPORTANT: "query" not "document"
#           Voyage trains separate spaces for queries vs docs
#           using "document" here would hurt performance
#     2. Normalize query vector to unit length
#        → enables cosine similarity via inner product
#           FAISS IndexFlatIP computes inner product
#           after normalization this equals cosine similarity
#     3. FAISS searches all 2887 vectors
#        returns indices of k most similar vectors
#     4. Use indices to look up text + metadata from chunks list
#        → chunks list and FAISS index share same ordering
#           index position 42 in FAISS = chunks[42]

class DenseRetriever:

    def __init__(self, chunks_path, faiss_index_path, voyage_api_key):
        """
        Load combined chunks and FAISS index.
        Both must be loaded together because FAISS returns
        position indices that map into the chunks list.
        """
        print("Loading Dense Retriever...")

        # Load chunks for text/metadata lookup
        with open(chunks_path) as f:
            self.chunks = json.load(f)

        # Load pre-built FAISS index
        # This is the vector index built during embedding
        self.index = faiss.read_index(faiss_index_path)

        # Initialize Voyage client for query embedding
        self.client = voyageai.Client(api_key=voyage_api_key)

        print(f"  ✓ Loaded {len(self.chunks)} chunks")
        print(f"  ✓ FAISS index: {self.index.ntotal} vectors, "
              f"dim={self.index.d}")


    def embed_query(self, query):
        """
        Embed query using Voyage.
        Uses input_type="query" — different from indexing
        which used input_type="document".

        Voyage's asymmetric embedding:
          document embedding → optimized to be found
          query embedding    → optimized to find documents
          Using matching types gives better retrieval
          than using "document" for both
        """
        response = self.client.embed(
            texts=[query],
            model="voyage-code-3",
            input_type="query"      # ← query not document
        )
        query_vec = np.array(
            [response.embeddings[0]],
            dtype='float32'
        )

        # Normalize to unit vector for cosine similarity
        faiss.normalize_L2(query_vec)
        return query_vec


    def retrieve(self, query, k=5):
        """
        Retrieve top-k most semantically similar chunks.

        Returns list of k dicts:
        [
          {
            'text':     chunk text (for debugging)
            'metadata': chunk metadata (for generation)
            'score':    cosine similarity score 0-1
                        higher = more similar
          }
        ]
        """
        # Step 1: embed query
        query_vec = self.embed_query(query)

        # Step 2: FAISS nearest neighbor search
        # distances: cosine similarity scores (0 to 1 after normalization)
        # indices:   positions in chunks list
        distances, indices = self.index.search(query_vec, k)

        # Step 3: look up chunks at returned positions
        return format_results(
            self.chunks,
            distances[0],
            indices[0]
        )


# Load dense retriever
dense_retriever = DenseRetriever(
    chunks_path      = DENSE_CHUNKS_PATH,
    faiss_index_path = DENSE_INDEX_PATH,
    voyage_api_key   = VOYAGE_API_KEY
)

print("\n✓ Dense Retriever ready")

Loading Dense Retriever...
  ✓ Loaded 4978 chunks
  ✓ FAISS index: 4978 vectors, dim=1024

✓ Dense Retriever ready


In [ ]:
# Sparse Retriever (SPLADE)
#
# Logic:
#   INDEXING (done during embedding, not here):
#     Each chunk text encoded into sparse dict
#     {token_id: weight} only storing non-zero entries
#     Represents importance of each vocabulary token
#     for this document
#
#   RETRIEVAL (done here):
#     1. Encode query with SPLADE → sparse dict
#        Same model, same encoding as documents
#        Query becomes {token_id: weight} dict
#     2. Compute sparse dot product between query
#        and every document sparse dict
#        Only multiply entries present in BOTH dicts
#        → efficient because most entries are zero
#     3. Sort scores → return top k
#
#   Why sparse dot product works:
#     If query has high weight for token "retrieve"
#     and document also has high weight for "retrieve"
#     their dot product is high → high relevance score
#     If query mentions "retrieve" but document doesn't
#     that token contributes 0 to the score
#     → naturally handles vocabulary mismatch better
#       than BM25 because SPLADE can assign weight to
#       semantically related tokens even if not exact match

class SparseRetriever:

    def __init__(self, chunks_path, model_id, hf_token):
        """
        Load combined SPLADE chunks and initialize SPLADE model
        for query encoding.

        Note: document sparse embeddings are pre-computed
        and stored in chunks_path. Only query needs to be
        encoded at retrieval time.
        """
        print("Loading Sparse Retriever...")

        # Load pre-computed sparse embeddings
        with open(chunks_path) as f:
            self.chunks = json.load(f)

        print(f"  ✓ Loaded {len(self.chunks)} sparse chunks")

        # Load SPLADE model for query encoding
        # Must use same model as used during indexing
        print(f"  Loading SPLADE model: {model_id}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, token=hf_token
        )
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_id,
            token=hf_token,
            tie_word_embeddings=False  # silences warning
        )

        # Use GPU if available — speeds up query encoding
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.model.to(self.device)
        self.model.eval()

        print(f"  ✓ SPLADE model loaded on {self.device}")


    def encode_query(self, query):
        """
        Encode query text into SPLADE sparse representation.
        Same process as document encoding during indexing.

        Returns dict: {token_id_str: weight_float}
        Only non-zero entries stored for efficiency.
        """
        inputs = self.tokenizer(
            [query],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            logits      = self.model(**inputs).logits

            # SPLADE core formula:
            # log1p(ReLU(logits)) → sparse weights
            # ReLU removes negative values (forces sparsity)
            # log1p compresses large values
            logits_relu = torch.log1p(torch.relu(logits))

            # Max pooling across sequence length
            # Each vocabulary token gets its maximum
            # activation across all input positions
            token_max, _ = torch.max(
                logits_relu * inputs.attention_mask.unsqueeze(-1),
                dim=1
            )

        # Extract non-zero entries only
        nonzero_indices = torch.nonzero(
            token_max[0]
        ).squeeze(-1).cpu().tolist()
        nonzero_weights = token_max[0][nonzero_indices].cpu().tolist()

        # Handle edge case: single non-zero entry
        if isinstance(nonzero_indices, int):
            nonzero_indices = [nonzero_indices]
        if isinstance(nonzero_weights, float):
            nonzero_weights = [nonzero_weights]

        return {
            str(idx): round(float(w), 4)
            for idx, w in zip(nonzero_indices, nonzero_weights)
        }


    def sparse_dot_product(self, query_sparse, doc_sparse):
        """
        Compute dot product between two sparse dicts.

        Only iterates over query tokens — documents without
        those tokens contribute 0 to the score automatically.

        This is mathematically equivalent to computing the
        full 30,522-dim dot product but much faster because
        most entries are zero.
        """
        score = 0.0
        for token_id, query_weight in query_sparse.items():
            if token_id in doc_sparse:
                score += query_weight * doc_sparse[token_id]
        return score


    def retrieve(self, query, k=5):
        """
        Retrieve top-k chunks using sparse dot product.

        Unlike dense retrieval there is no FAISS index —
        we compute dot product against all documents directly.
        This is feasible because:
          1. Sparse dicts are small (~150-350 entries each)
          2. Dot product only iterates over query tokens
          3. 2887 documents is small enough for linear scan

        Returns same format as DenseRetriever.retrieve()
        """
        # Step 1: encode query to sparse dict
        query_sparse = self.encode_query(query)

        print(f"  Query encoded: {len(query_sparse)} active tokens")

        # Step 2: score all documents
        # Linear scan — no FAISS needed for sparse
        scores = []
        for i, chunk in enumerate(self.chunks):
            doc_sparse = chunk.get('sparse_embedding', {})
            score      = self.sparse_dot_product(
                query_sparse, doc_sparse
            )
            scores.append((score, i))

        # Step 3: sort by score descending → top k
        scores.sort(key=lambda x: x[0], reverse=True)
        top_k = scores[:k]

        top_scores  = [s for s, _ in top_k]
        top_indices = [i for _, i in top_k]

        return format_results(self.chunks, top_scores, top_indices)


# Load sparse retriever
# Note: requires GPU for reasonable query encoding speed
sparse_retriever = SparseRetriever(
    chunks_path = SPLADE_CHUNKS_PATH,
    model_id    = 'naver/splade-v3',
    hf_token    = HF_TOKEN
)

print("\n✓ Sparse Retriever ready")

Loading Sparse Retriever...
  ✓ Loaded 5005 sparse chunks
  Loading SPLADE model: naver/splade-v3...


config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: naver/splade-v3
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ SPLADE model loaded on cpu

✓ Sparse Retriever ready


In [ ]:
# BM25 Retriever
#
# Logic:
#   INDEXING (done during embedding, not here):
#     All chunk texts tokenized into word lists
#     BM25Okapi builds inverted index:
#       term frequencies per document
#       IDF weights per term
#       average document length
#     Saved as pickle file
#
#   RETRIEVAL (done here):
#     1. Tokenize query into word list
#        Same tokenization as used during indexing
#        → CRITICAL: must match exactly
#           different tokenization = wrong scores
#     2. BM25 formula scores all documents at once
#        Score depends on:
#          TF: how often query term appears in document
#          IDF: how rare is this term across all documents
#          Length norm: penalizes very long documents
#     3. argsort scores → top k indices
#     4. Look up chunks at those positions
#
#   Key difference from dense/sparse:
#     BM25 has NO neural encoding
#     Pure statistical scoring on exact word matches
#     Fast, no GPU needed, no API calls
#     Weakness: "retrieve" and "get" are different words
#               BM25 cannot know they mean the same thing

class BM25Retriever:

    def __init__(self, index_path, chunks_path):
        """
        Load BM25 index (pickle) and chunks (JSON).

        Two files needed because:
          BM25 index: Python object with corpus statistics
                      cannot be stored as JSON
                      stores term frequencies, IDF weights,
                      document lengths, average doc length
          chunks JSON: text and metadata for each document
                       needed to return actual content

        CRITICAL: chunks must be in same order as when
                  BM25 index was built — position 0 in
                  chunks = document 0 in BM25 index
        """
        print("Loading BM25 Retriever...")

        # Load BM25 index — restores complete Python object
        with open(index_path, 'rb') as f:
            self.bm25 = pickle.load(f)

        # Load chunks for text/metadata lookup
        with open(chunks_path) as f:
            self.chunks = json.load(f)

        print(f"  ✓ BM25 index loaded")
        print(f"    corpus_size: {self.bm25.corpus_size}")
        print(f"    avgdl:       {self.bm25.avgdl:.1f} tokens")
        print(f"  ✓ Chunks loaded: {len(self.chunks)}")

        # Verify counts match
        assert self.bm25.corpus_size == len(self.chunks), (
            f"BM25 index size ({self.bm25.corpus_size}) "
            f"doesn't match chunks count ({len(self.chunks)}). "
            f"Index and chunks are out of sync."
        )


    def retrieve(self, query, k=5):
        """
        Retrieve top-k chunks using BM25 scoring.

        BM25 score formula for each document d:
          score(q, d) = Σ IDF(t) × TF(t,d) × (k1+1)
                         t∈q       ─────────────────────────
                                   TF(t,d) + k1×(1-b+b×|d|/avgdl)

          Where:
            t     = each term in query q
            IDF   = log((N - df + 0.5) / (df + 0.5))
                    N  = total documents
                    df = documents containing term t
            TF    = frequency of term t in document d
            |d|   = length of document d in tokens
            avgdl = average document length across corpus
            k1,b  = tuning parameters (k1=1.5, b=0.75 default)

          Rare terms (high IDF) contribute more to score
          Frequent terms in document (high TF) boost score
          Long documents are penalized by length normalization
        """
        # Step 1: tokenize query
        # MUST use same tokenizer as indexing
        tokenized_query = tokenize_for_bm25(query)

        # Step 2: BM25 scores ALL documents at once
        # Returns numpy array of length = corpus_size
        scores = self.bm25.get_scores(tokenized_query)

        # Step 3: get indices of top-k highest scores
        # argsort returns indices that would sort array ascending
        # [::-1] reverses to descending
        # [:k] takes top k
        top_k_indices = np.argsort(scores)[::-1][:k]
        top_k_scores  = scores[top_k_indices]

        return format_results(
            self.chunks,
            top_k_scores,
            top_k_indices
        )


# Load BM25 retriever
bm25_retriever = BM25Retriever(
    index_path  = BM25_INDEX_PATH,
    chunks_path = BM25_CHUNKS_PATH
)

print("\n✓ BM25 Retriever ready")

Loading BM25 Retriever...


Error during conversion: AttributeError("'str' object has no attribute 'decode'")
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The

  ✓ BM25 index loaded
    corpus_size: 5005
    avgdl:       37.3 tokens
  ✓ Chunks loaded: 5005

✓ BM25 Retriever ready


In [ ]:
#Construct the retrieval tuning policy set
import json
import os
import random
import time

# ── Load the 470 remaining managed policies ───────────────────────────
managed_path = os.path.join(
    DRIVE_BASE, "data/processed/managed_policies.json"
)
with open(managed_path) as f:
    managed = json.load(f)

print(f"Managed policies loaded: {len(managed)}")

# ── Remove the 30 ground truth managed policies ───────────────────────
gt_path = os.path.join(
    DRIVE_BASE, "data/evaluation/evaluation_ground_truth.json"
)
with open(gt_path) as f:
    gt = json.load(f)

gt_names = {
    p["policy_name"]
    for p in gt
    if p["source"] == "aws_managed"
}

managed = [p for p in managed if p["policy_name"] not in gt_names]
print(f"After removing {len(gt_names)} GT policies: {len(managed)} remaining")

# ── Core target services — same as your retrieval corpus ─────────────
CORE_SERVICES = {
    's3', 'lambda', 'dynamodb', 'ec2', 'ecs', 'ecr',
    'sqs', 'sns', 'logs', 'cloudwatch', 'events', 'kms',
    'secretsmanager', 'ssm', 'iam', 'sts', 'codedeploy',
    'codepipeline', 'glue', 'states', 'kinesis', 'firehose',
    'rds', 'eks', 'kafka', 'redshift', 'athena', 'codebuild',
    'xray', 'cloudformation', 'apigateway', 'elasticloadbalancing',
    'rekognition', 'ses', 'stepfunctions', 'sagemaker'
}

# ── Niche services to penalize ────────────────────────────────────────
NICHE_SERVICES = {
    'cassandra', 'kinesisanalytics', 'rum', 'evidently',
    'imagebuilder', 'resource-explorer-2', 'application-signals',
    'fsx', 'sdb', 'outposts', 's3-outposts', 'drs',
    'codeconnections', 'codestar-connections', 'guardduty',
    'pi', 'elasticmapreduce', 'servicecatalog', 'appstream',
    'aoss', 'synthetics', 'cloudfront', 'monitron',
    'thinclient', 'workspaces', 'workspaces-web', 'aiops',
    'datasync', 'dms', 'mgn', 'codecommit', 'codeguru',
    'devopsguru', 'inspector', 'inspector2', 'chatbot',
    'codestar-notifications', 'databrew', 'mobiletargeting',
    'geo', 'opsworks', 'elasticbeanstalk', 'organizations',
    'account', 'sso', 'sso-directory', 'oam', 'tag',
    'application-autoscaling', 'autoscaling'
}

def get_specific_actions(policy):
    return [
        a for a in policy.get("actions_used", [])
        if isinstance(a, str) and ":" in a and "*" not in a
    ]

def has_service_wildcard(policy):
    return any(
        a.endswith(":*")
        for a in policy.get("actions_used", [])
        if isinstance(a, str)
    )

def get_services(policy):
    return {
        a.split(":")[0].lower()
        for a in policy.get("actions_used", [])
        if isinstance(a, str) and ":" in a
    }

def score_for_dev_set(policy):
    """
    Lightweight scoring to prefer mainstream, well-covered policies.
    Simpler than ground truth scoring — just needs to filter out niche.
    """
    specific = get_specific_actions(policy)
    services = get_services(policy)

    # ── Hard filters ─────────────────────────────────────────
    if len(specific) < 3:
        return 0
    if len(specific) > 15:
        return 0
    if has_service_wildcard(policy):
        return 0

    # Must cover at least one core service
    if not services & CORE_SERVICES:
        return 0

    # ── Score ─────────────────────────────────────────────────
    score = 10

    # Bonus for core service coverage
    score += len(services & CORE_SERVICES) * 4

    # Bonus for sweet spot action count
    n = len(specific)
    if 4 <= n <= 10:
        score += 4
    elif 11 <= n <= 15:
        score += 2

    # Penalty for niche services
    score -= len(services & NICHE_SERVICES) * 5

    # Penalty for being dominated by niche services
    niche_ratio = len(services & NICHE_SERVICES) / max(len(services), 1)
    if niche_ratio > 0.5:
        score -= 10  # majority niche — strong penalty

    # Penalty for spanning too many services
    # (harder to write a focused query)
    if len(services) > 5:
        score -= (len(services) - 5) * 2

    return max(score, 0)


# ── Score and filter ──────────────────────────────────────────────────
scored = [
    (p, score_for_dev_set(p))
    for p in managed
]
scored = [(p, s) for p, s in scored if s > 0]
scored.sort(key=lambda x: x[1], reverse=True)

print(f"Policies passing filters:  {len(scored)}")
print(f"\nTop 10 candidates by score:")
for p, s in scored[:10]:
    svcs = sorted(get_services(p) & CORE_SERVICES)
    n    = len(get_specific_actions(p))
    print(f"  score={s:>3}  {p['policy_name']:<55} "
          f"services={svcs}  actions={n}")

# ── Sample 100 from top 200 to add diversity ──────────────────────────
# Taking strictly top 100 would bias toward similar high-scoring policies
# Sampling from top 200 adds variety while still avoiding niche ones
top_candidates = [p for p, s in scored[:200]]

random.seed(42)
dev_sample = random.sample(
    top_candidates,
    min(100, len(top_candidates))
)
print(f"\nDev set size: {len(dev_sample)}")

# ── Verify service distribution ───────────────────────────────────────
from collections import Counter
service_counts = Counter()
for p in dev_sample:
    for svc in get_services(p):
        service_counts[svc] += 1

print(f"\nService coverage in dev set (top 15):")
for svc, count in service_counts.most_common(15):
    marker = "✓" if svc in CORE_SERVICES else "~"
    print(f"  {marker} {svc:<25} {count:>3} policies")

# ── Generate queries using same GPT-4o prompt as ground truth ─────────
SYSTEM_PROMPT = """You are a developer writing a task description
that will be used to generate an AWS IAM policy. Given an IAM policy,
write a clear, specific task description that captures exactly what
permissions are needed and why.

RULES:
- Do NOT mention specific AWS service names
- Do NOT mention specific IAM action names
- Write as a statement, not a question
- Do not wrap the query in quotation marks
- Be specific about WHAT the service/role needs to do
- Keep to 1-3 sentences
- If the policy includes role-passing permissions, do not mention
  them as a separate task
- Focus only on primary functional tasks"""

dev_set = []

print("\nGenerating queries for dev set...")
for i, policy in enumerate(dev_sample):
    specific_actions = get_specific_actions(policy)
    services = sorted({
        a.split(":")[0].lower()
        for a in specific_actions
    })

    context = f"AWS managed policy name: {policy['policy_name']}\n"
    context += f"Specific actions granted: {', '.join(specific_actions)}\n"

    if len(services) > 2:
        context += (
            f"This policy covers {len(services)} service areas: "
            f"{', '.join(services)}. Mention all distinct functional tasks."
        )

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": context +
                 "\n\nWrite a task description for this policy."}
            ],
            temperature=0.2,
            max_tokens=200
        )
        query = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Error on {policy['policy_name']}: {e}")
        query = None

    if query:
        dev_set.append({
            "policy_name": policy["policy_name"],
            "source":      "aws_managed_dev",
            "actions":     specific_actions,
            "services":    services,
            "query":       query,
        })
        print(f"  [{i+1:03d}] {policy['policy_name']}")
        print(f"         → {query[:80]}...")

    time.sleep(0.3)

print(f"\n✓ Dev set constructed: {len(dev_set)} policies with queries")

# ── Save dev set ──────────────────────────────────────────────────────
dev_path = os.path.join(
    DRIVE_BASE, "data/processed/retrieval_dev_set.json"
)
with open(dev_path, "w") as f:
    json.dump(dev_set, f, indent=2)

print(f"✓ Saved to: {dev_path}")

Managed policies loaded: 500
After removing 30 GT policies: 473 remaining
Policies passing filters:  120

Top 10 candidates by score:
  score= 36  AWSCodeDeployRoleForECSLimited                          services=['cloudwatch', 'ecs', 'elasticloadbalancing', 'iam', 'lambda', 's3', 'sns']  actions=15
  score= 32  CloudWatchAgentAdminPolicy                              services=['cloudwatch', 'ec2', 'logs', 'ssm', 'xray']  actions=15
  score= 32  AWS-SSM-RemediationAutomation-AdministrationRolePolicy  services=['iam', 'kms', 's3', 'ssm', 'sts']  actions=12
  score= 32  AmazonWorkMailReadOnlyAccess                            services=['cloudwatch', 'iam', 'lambda', 'logs', 'ses']  actions=4
  score= 30  AmazonRedshiftReadOnlyAccess                            services=['cloudwatch', 'ec2', 'redshift', 'sns']  actions=9
  score= 27  AmazonKinesisAnalyticsReadOnly                          services=['cloudwatch', 'firehose', 'iam', 'kinesis', 'logs']  actions=9
  score= 26  AmazonEC2SpotFleetT

In [ ]:
# RRF Hybrid Retriever

from collections import defaultdict

def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Combine multiple ranked lists using RRF.

    Args:
        ranked_lists: list of lists of chunk indices
                      each inner list is one retriever's
                      ranking from most to least relevant
        k:            smoothing constant (default 60
                      from Cormack et al. 2009)

    Returns:
        list of (chunk_idx, rrf_score) sorted by score desc

    Formula:
        RRF(doc) = Σ 1/(k + rank_i)

        A document ranked 1st contributes 1/(60+1) = 0.0164
        A document ranked 10th contributes 1/(60+10) = 0.0143

        Documents appearing in multiple lists accumulate
        scores — consistency across methods is rewarded
    """
    scores = defaultdict(float)

    for ranked_list in ranked_lists:
        for rank, chunk_idx in enumerate(ranked_list):
            scores[chunk_idx] += 1.0 / (k + rank + 1)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )


class HybridRetriever:
    """
    RRF hybrid retrieval combining Dense + BM25.
    Optionally includes SPLADE as third method.

    Design choice rationale:
      Dense: semantic understanding of natural language queries
      BM25:  exact keyword matching for technical terms
      RRF:   rewards consistency across methods
             a document found relevant by both Dense and BM25
             is almost certainly truly relevant
    """

    def __init__(self, dense_retriever, bm25_retriever,
                 sparse_retriever=None,
                 include_splade=False,
                 k=60,
                 weights=None):
        """
        Args:
            dense_retriever:   DenseRetriever instance
            bm25_retriever:    BM25Retriever instance
            sparse_retriever:  SparseRetriever instance (optional)
            include_splade:    whether to include SPLADE in fusion
            k:                 RRF smoothing constant
        """
        self.dense   = dense_retriever
        self.bm25    = bm25_retriever
        self.sparse  = sparse_retriever
        self.include_splade = include_splade
        self.k       = k

        if weights is None:
            self.weights = [1.0, 1.0, 1.0]
        else:
            self.weights = weights

        print(f"  Weights: dense={self.weights[0]} "
              f"bm25={self.weights[1]} "
              f"splade={self.weights[2] if len(self.weights) > 2 else 'N/A'}")

        # Use dense chunks as the unified chunk lookup
        # All retrievers share same chunk ordering
        # so we can use any retriever's chunks for lookup
        self.chunks  = dense_retriever.chunks

        methods = ['Dense', 'BM25']
        if include_splade and sparse_retriever:
            methods.append('SPLADE')
        print(f"HybridRetriever initialized")
        print(f"  Methods: {' + '.join(methods)}")
        print(f"  RRF k:   {k}")


    def retrieve(self, query, k=5, candidate_pool=50):
        """
        Hybrid retrieval via RRF.

        Args:
            query:          natural language query string
            k:              number of results to return
            candidate_pool: how many candidates each retriever
                           fetches before fusion
                           larger pool = better fusion
                           at cost of more computation
                           recommend: k*10 = 50 for k=5

        Logic:
            1. Each retriever fetches candidate_pool results
               independently (50 each)
            2. RRF scores every document that appeared in
               any retriever's list
            3. Documents appearing in multiple lists get
               higher scores (consistency rewarded)
            4. Return top k from combined ranking
        """

        # ── Step 1: get candidates from each retriever ────────
        dense_results  = self.dense.retrieve(query, k=candidate_pool)
        bm25_results   = self.bm25.retrieve(query,  k=candidate_pool)

        # Extract ranked lists of indices
        # Each list: [most_relevant_idx, ..., least_relevant_idx]
        dense_ranking  = [
            self._get_chunk_idx(r, 'dense')
            for r in dense_results
        ]
        bm25_ranking   = [
            self._get_chunk_idx(r, 'bm25')
            for r in bm25_results
        ]
        """

        ranked_lists   = [dense_ranking, bm25_ranking]
        method_names   = ['dense', 'bm25']

        # Optionally include SPLADE
        if self.include_splade and self.sparse:
            sparse_results  = self.sparse.retrieve(
                query, k=candidate_pool
            )
            sparse_ranking  = [
                self._get_chunk_idx(r, 'sparse')
                for r in sparse_results
            ]
            ranked_lists.append(sparse_ranking)
            method_names.append('splade')

        # ── Step 2: RRF fusion ────────────────────────────────
        combined = reciprocal_rank_fusion(ranked_lists, k=self.k)
"""
#new:
        ranked_lists = [dense_ranking, bm25_ranking]
        method_names = ['dense', 'bm25']
        active_weights = [self.weights[0], self.weights[1]]

        # Optionally include SPLADE
        if self.include_splade and self.sparse:
            sparse_results = self.sparse.retrieve(
                query, k=candidate_pool
            )
            sparse_ranking = [
                self._get_chunk_idx(r, 'sparse')
                for r in sparse_results
            ]
            ranked_lists.append(sparse_ranking)
            method_names.append('splade')
            active_weights.append(
                self.weights[2] if len(self.weights) > 2 else 1.0
            )

        # ── Step 2: Weighted RRF fusion ───────────────────────
        combined = weighted_rrf(ranked_lists, active_weights, k=self.k)

        # ── Step 3: build results for top k ───────────────────
        results = []
        for chunk_idx, rrf_score in combined[:k]:

            chunk = self.chunks[chunk_idx]

            # Track which methods found this chunk
            # and at what rank — useful for analysis
            method_ranks = {}
            for method_name, ranking in zip(
                method_names, ranked_lists
            ):
                if chunk_idx in ranking:
                    method_ranks[method_name] = (
                        ranking.index(chunk_idx) + 1
                    )
                else:
                    method_ranks[method_name] = None

            results.append({
                'text':         chunk['text'],
                'metadata':     chunk['metadata'],
                'rrf_score':    rrf_score,
                'method_ranks': method_ranks
            })

        return results


    def _get_chunk_idx(self, result, method):
        """
        Find the position of a result in the combined chunks list.
        Different retrievers use different chunk lists internally
        but all share the same underlying chunk ordering.
        """
        # Match by action or policy_name in metadata
        metadata = result['metadata']

        if metadata['type'] == 'iam_action':
            target_action = metadata.get('action')
            for i, chunk in enumerate(self.chunks):
                if chunk['metadata'].get('action') == target_action:
                    return i

        else:
            target_name = metadata.get('policy_name')
            for i, chunk in enumerate(self.chunks):
                if chunk['metadata'].get('policy_name') == target_name:
                    return i

        return -1  # not found — should not happen


# ── Load hybrid retrievers ─────────────────────────────────────────────

# Primary: Dense + BM25 (recommended)
hybrid_dense_bm25 = HybridRetriever(
    dense_retriever  = dense_retriever,
    bm25_retriever   = bm25_retriever,
    include_splade   = False,
    k                = 60,
    weights          = [1.0, 0.5]
)

# Secondary: Dense + BM25 + SPLADE (for ablation)
hybrid_all_three = HybridRetriever(
    dense_retriever  = dense_retriever,
    bm25_retriever   = bm25_retriever,
    sparse_retriever = sparse_retriever,
    include_splade   = True,
    k                = 60,
    weights          = [1.0, 0.3, 0.3]
)

print("\n✓ Hybrid Retrievers ready")

  Weights: dense=1.0 bm25=0.5 splade=N/A
HybridRetriever initialized
  Methods: Dense + BM25
  RRF k:   60
  Weights: dense=1.0 bm25=0.3 splade=0.3
HybridRetriever initialized
  Methods: Dense + BM25 + SPLADE
  RRF k:   60

✓ Hybrid Retrievers ready


In [ ]:
# ── FINAL retrieval function ─────────────────────
# Dense only for both actions and policies
# Based on retrieval analysis showing BM25 degrades
# action quality for IAM domain

def retrieve_for_generation(query,
                             dense_retriever,
                             k_actions,
                             k_policies):
    """
    Dense-only type-aware retrieval.
    Actions: Dense k=5
    Policies: Dense k=2
    Total: 7 chunks
    """
    dense_results  = dense_retriever.retrieve(query, k=100)

    dense_actions  = [r for r in dense_results
                      if r['metadata']['type'] == 'iam_action']
    dense_policies = [r for r in dense_results
                      if r['metadata']['type'] == 'iam_policy']

    results = []
    for r in dense_actions[:k_actions]:
        r['source'] = 'action_dense'
        results.append(r)
    for r in dense_policies[:k_policies]:
        r['source'] = 'policy_dense'
        results.append(r)

    results = apply_companion_rules(results, dense_retriever)
    return results


# ── Ablation baseline kept for results table ───────────────
def retrieve_for_generation_rrf(query,
                                 dense_retriever,
                                 bm25_retriever,
                                 k_actions,
                                 k_policies):
    """RRF(Dense+BM25) ablation baseline."""
    dense_results = dense_retriever.retrieve(query, k=100)
    bm25_results  = bm25_retriever.retrieve(query,  k=100)

    dense_actions  = [r for r in dense_results
                      if r['metadata']['type'] == 'iam_action']
    dense_policies = [r for r in dense_results
                      if r['metadata']['type'] == 'iam_policy']
    bm25_actions   = [r for r in bm25_results
                      if r['metadata']['type'] == 'iam_action']

    def find_idx(result, chunks):
        metadata = result['metadata']
        target   = metadata.get('action') or \
                   metadata.get('policy_name')
        field    = 'action' if metadata['type'] == 'iam_action' \
                   else 'policy_name'
        for i, c in enumerate(chunks):
            if c['metadata'].get(field) == target:
                return i
        return -1

    dense_ranking = [find_idx(r, dense_retriever.chunks)
                     for r in dense_actions]
    bm25_ranking  = [find_idx(r, dense_retriever.chunks)
                     for r in bm25_actions]
    dense_ranking = [i for i in dense_ranking if i >= 0]
    bm25_ranking  = [i for i in bm25_ranking  if i >= 0]

    action_combined = reciprocal_rank_fusion(
        [dense_ranking, bm25_ranking], k=60
    )

    results = []
    for chunk_idx, rrf_score in action_combined[:k_actions]:
        chunk = dense_retriever.chunks[chunk_idx]
        results.append({
            'text':     chunk['text'],
            'metadata': chunk['metadata'],
            'score':    rrf_score,
            'source':   'action_rrf'
        })
    for r in dense_policies[:k_policies]:
        results.append({
            'text':     r['text'],
            'metadata': r['metadata'],
            'score':    r['score'],
            'source':   'policy_dense'
        })
    results = apply_companion_rules(results, dense_retriever)
    return results

print("✓ retrieve_for_generation() defined (Dense only)")
print("✓ retrieve_for_generation_rrf() defined (ablation)")

✓ retrieve_for_generation() defined (Dense only)
✓ retrieve_for_generation_rrf() defined (ablation)


In [ ]:
def decompose_query_v2(query, openai_client):
    """
    Improved query decomposition that explicitly identifies
    implicit cross-service dependencies.

    Key improvement over v1:
      Explicitly asks for implicit requirements like
      CloudWatch Logs for Lambda, GetAuthorizationToken
      for ECR, etc.
    """
    prompt = f"""You are an AWS IAM expert.

Given this AWS task:
"{query}"

Identify ALL AWS services and permission types needed,
including IMPLICIT requirements that are not mentioned
in the task but are always needed.

Please follow these rules when identifying services:
1. Maximum 4 sub-queries total (explicit + implicit combined)
2. NO duplicate services — each service appears at most ONCE
3. Sub-query phrases should be SPECIFIC enough to find exact actions
   BAD:  "Lambda permissions"
   GOOD: "Lambda invoke function event source SQS trigger"
   BAD:  "EC2 manage instances"
   GOOD: "EC2 describe instance status run command"
   BAD:  "KMS decrypt"
   GOOD: "KMS decrypt describe key"
   BAD:  "DynamoDB read stream"
   GOOD: "DynamoDB stream get records get shard iterator describe stream list streams"

4. Only add implicit services if they are UNIVERSALLY required:
   - Lambda ALWAYS needs: "CloudWatch Logs create log group stream put events"
   - ECR pull ALWAYS needs: "ECR get authorization token"
   DO NOT add KMS, IAM, ELB unless explicitly mentioned in task

5. Each reason must name the SPECIFIC actions needed, not just the service

Return a JSON object:
{{
  "explicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "needs action1, action2, action3 for this task"
    }}
  ],
  "implicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "always needed: action1, action2, action3"
    }}
  ]
}}

Return ONLY the JSON, nothing else."""

    response = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=400
    )

    try:
        text = response.choices[0].message.content.strip()
        if '```' in text:
            text = text.split('```')[1].split('```')[0]
            if text.startswith('json'):
                text = text[4:]
        result = json.loads(text.strip())

        # Combine explicit and implicit into flat list
        all_sub_queries = []
        for item in result.get('explicit_services', []):
            all_sub_queries.append({
                'query':    item['query'],
                'type':     'explicit',
                'service':  item['service']
            })
        for item in result.get('implicit_services', []):
            all_sub_queries.append({
                'query':    item['query'],
                'type':     'implicit',
                'service':  item['service'],
                'reason':   item.get('reason', '')
            })

        return all_sub_queries

    except Exception as e:
        print(f"  Decomposition failed: {e}")
        return [{'query': query, 'type': 'original', 'service': 'unknown'}]


def retrieve_for_generation_decomposed_v2(query,
                                           dense_retriever,
                                           openai_client,
                                           k_per_subquery=2,
                                           k_policies=2,
                                          score_threshold=0.45):
    """
    Improved decomposed retrieval.

    Improvements over v1:
      1. Explicitly identifies implicit services
      2. Larger k_per_subquery for better coverage
      3. Shows which actions came from which sub-query
      4. Deduplicates across sub-queries
    """
    sub_queries = decompose_query_v2(query, openai_client)

    print(f"  Decomposed into {len(sub_queries)} sub-queries:")
    for sq in sub_queries:
        tag = f"[{sq['type']}]"
        print(f"    {tag:<12} {sq['service']}: {sq['query']}")

    seen_actions = set()
    all_actions  = []

    for sub_query_info in sub_queries:
        sub_query = sub_query_info['query']
        results   = dense_retriever.retrieve(sub_query, k=30)
        actions   = [r for r in results
                     if r['metadata']['type'] == 'iam_action']

        added = 0
        for r in actions:
            action = r['metadata']['action']
            clean  = action.split(' [')[0]

            if r['score'] < score_threshold:
              break

            if clean not in seen_actions and added < k_per_subquery:
                seen_actions.add(clean)
                r['source']    = f"action_{sub_query_info['type']}"
                r['sub_query'] = sub_query
                all_actions.append(r)
                added += 1

    # NEW — one policy per sub-query, deduplicated
    seen_policies = set()
    all_policies  = []

    for sub_query_info in sub_queries:
        sub_query   = sub_query_info['query']
        results_sq  = dense_retriever.retrieve(sub_query, k=30)
        policies_sq = [r for r in results_sq
                      if r['metadata']['type'] == 'iam_policy']

        for r in policies_sq:
            name = r['metadata'].get('policy_name', '')
            n_actions = len(r['metadata'].get('actions_used', []))
            if name not in seen_policies and n_actions <= 15:
                seen_policies.add(name)
                r['source'] = 'policy_decomposed'
                all_policies.append(r)
                break

    results = all_actions.copy()
    for r in all_policies[:k_policies]:
        results.append(r)

    print(f"  Total retrieved: {len(results)} chunks "
          f"({len(all_actions)} actions + "
          f"{len(results)-len(all_actions)} policies)")

    results = apply_companion_rules(results, dense_retriever)
    return results

In [ ]:
def decompose_query_gpt4o(query, openai_client):
    """
    Use GPT-4o as decomposer for higher precision.
    GPT-3.5 remains the generator.

    """
    prompt = f"""You are an AWS IAM expert with deep knowledge
of AWS service dependencies and implicit permission requirements.

Given this AWS task:
"{query}"

Your job is to decompose it into HIGH-PRECISION retrieval sub-queries
that recover BOTH primary actions and companion actions commonly required
for a working IAM policy. Identify ALL AWS services and permission types needed,
including IMPLICIT requirements that are not mentioned
in the task but are always needed.

Please follow these rules when identifying services:
1. Maximum 4 sub-queries total (explicit + implicit combined)
2. NO duplicate services — each service appears at most ONCE
3. Sub-query phrases must be ACTION-BUNDLE oriented and specific enough to retrieve exact IAM actions.
BAD:  "KMS decrypt"
GOOD: "KMS decrypt describe key"
BAD:  "DynamoDB read stream"
GOOD: "DynamoDB stream get records get shard iterator describe stream list streams"

4. Only add implicit services if they are UNIVERSALLY required:
   - Lambda ALWAYS needs: "CloudWatch Logs create log group stream put events"
   - ECR pull ALWAYS needs: "ECR get authorization token"
   DO NOT add KMS, IAM, ELB unless explicitly mentioned in task
5. Implicit service rules — ONLY add these when the named trigger is present:\n
   - 'logs' sub-query: ONLY if task mentions Lambda, ECS, or API Gateway\n
   - 'ecr' sub-query:  ONLY if task mentions pulling/running container images\n
   - 'kms' sub-query:  ONLY if task explicitly mentions encryption or KMS\n
   - 'sts' sub-query:  ONLY if task mentions assuming a role or cross-account\n
   DO NOT add implicit services for S3-only, SNS-only, or SQS-only tasks.\n

6. For services that have both a TRIGGERING/PUBLISH action and a
   CONFIGURATION/PERMISSION action, you MUST include BOTH in the query phrase:
   BAD:  "EventBridge put events"
   GOOD: "EventBridge put events put rule put targets configure"
   BAD:  "Lambda invoke function"
   GOOD: "Lambda invoke function add permission event source"
   This applies ONLY to:
  - EventBridge: put rule + put targets
  - Lambda: invoke function + add permission (only when invoked by another AWS service)
  Do NOT add SNS:AddPermission, SQS:AddPermission, or API Gateway write/configure actions
  unless the task explicitly describes resource-policy management or endpoint configuration changes.

Return a JSON object:
{{
  "explicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "needs action1, action2, action3 for this task"
    }}
  ],
  "implicit_services": [
    {{
      "service": "aws_service_prefix",
      "query": "specific action-level search phrase",
      "reason": "always needed: action1, action2, action3"
    }}
  ]
}}

Return ONLY valid JSON."""

    response = openai_client.chat.completions.create(
        model="gpt-4o",              # ← stronger decomposer
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )

    try:
        text = response.choices[0].message.content.strip()
        if '```' in text:
            text = text.split('```')[1].split('```')[0]
            if text.startswith('json'):
                text = text[4:]
        result = json.loads(text.strip())

        all_sub_queries = []
        for item in result.get('explicit_services', []):
            all_sub_queries.append({
                'query':   item['query'],
                'type':    'explicit',
                'service': item['service'],
                'reason':  item.get('reason', '')
            })
        for item in result.get('implicit_services', []):
            all_sub_queries.append({
                'query':   item['query'],
                'type':    'implicit',
                'service': item['service'],
                'reason':  item.get('reason', '')
            })

        return all_sub_queries

    except Exception as e:
        print(f"  Decomposition failed: {e}")
        return [{'query': query, 'type': 'original',
                 'service': 'unknown', 'reason': ''}]


def retrieve_for_generation_decomposed_v3(query,
                                           dense_retriever,
                                           openai_client,
                                           k_per_subquery=2,
                                           k_policies=2,
                                          score_threshold=0.55):
    """
    Decomposed retrieval using GPT-4o as decomposer.
    GPT-3.5 remains the generator — only decomposition upgraded.
    """
    sub_queries = decompose_query_gpt4o(query, openai_client)

    print(f"  GPT-4o decomposed into {len(sub_queries)} sub-queries:")
    for sq in sub_queries:
        tag = f"[{sq['type']}]"
        print(f"    {tag:<12} {sq['service']}: {sq['query']}")
        print(f"               reason: {sq['reason'][:60]}")

    seen_actions = set()
    all_actions  = []

    for sub_query_info in sub_queries:
        sub_query = sub_query_info['query']
        results   = dense_retriever.retrieve(sub_query, k=30)
        actions   = [r for r in results
                     if r['metadata']['type'] == 'iam_action']

        added = 0
        for r in actions:
            action = r['metadata']['action']
            clean  = action.split(' [')[0]

            if r['score'] < score_threshold:
                break
            if clean not in seen_actions and added < k_per_subquery:
                seen_actions.add(clean)
                r['source']    = f"action_{sub_query_info['type']}"
                r['sub_query'] = sub_query
                all_actions.append(r)
                added += 1

    # NEW — one policy per sub-query, deduplicated
    seen_policies = set()
    all_policies  = []

    for sub_query_info in sub_queries:
        sub_query   = sub_query_info['query']
        results_sq  = dense_retriever.retrieve(sub_query, k=30)
        policies_sq = [r for r in results_sq
                      if r['metadata']['type'] == 'iam_policy']

        for r in policies_sq:
            name = r['metadata'].get('policy_name', '')
            n_actions = len(r['metadata'].get('actions_used', []))
            if name not in seen_policies and n_actions <= 15:
                seen_policies.add(name)
                r['source'] = 'policy_decomposed'
                all_policies.append(r)
                break

    results = all_actions.copy()
    for r in all_policies[:k_policies]:
        results.append(r)

    print(f"  Total retrieved: {len(results)} chunks "
          f"({len(all_actions)} actions + "
          f"{len(results)-len(all_actions)} policies)")

    results = apply_companion_rules(results, dense_retriever)
    return results

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Retrieval recall measurement in isolation
# ═══════════════════════════════════════════════════════════════════════

import json
import os
from collections import Counter

# Load dev set
dev_path = os.path.join(
    DRIVE_BASE, "data/processed/retrieval_dev_set.json"
)
with open(dev_path) as f:
    dev_set = json.load(f)

print(f"Dev set loaded: {len(dev_set)} policies")

def normalize_action_name(action):
    """Strip [permission only] and other suffixes."""
    return action.split(' [')[0].strip()

def extract_actions(policy_str):
    """Extract all actions from generated policy JSON."""
    try:
        clean = policy_str
        if '```json' in clean:
            clean = clean.split('```json')[1].split('```')[0]
        elif '```' in clean:
            clean = clean.split('```')[1].split('```')[0]

        policy  = json.loads(clean.strip())
        actions = []
        for statement in policy.get('Statement', []):
            raw = statement.get('Action', [])
            if isinstance(raw, str):
                raw = [raw]
            actions.extend(raw)

        return list(set(actions)), policy

    except json.JSONDecodeError:
        return [], None

def measure_retrieval_recall(gt_actions, retrieved_chunks):
    """
    Measure recall using only action chunks — ignores policy chunks.
    This prevents Dense from 'cheating' by retrieving the exact
    policy document from the corpus on dev set queries.
    Gives a fairer comparison between Dense and Decomposed.
    """
    retrieved_actions = set()

    for chunk in retrieved_chunks:
        # Only count action chunks — not policy chunks
        if chunk["metadata"]["type"] == "iam_action":
            action = normalize_action_name(
                chunk["metadata"].get("action", "")
            )
            retrieved_actions.add(action.lower())

    gt_set  = {normalize_action_name(a).lower() for a in gt_actions}
    covered = gt_set & retrieved_actions
    missing = gt_set - retrieved_actions

    return {
        "retrieval_recall":       len(covered) / len(gt_set) if gt_set else 0,
        "covered":                sorted(covered),
        "missing_from_retrieval": sorted(missing),
        "n_gt":                   len(gt_set),
        "n_covered":              len(covered),
    }

def run_retrieval_evaluation(dev_set, retrieval_fn, label,
                              verbose=False):
    """
    Measure retrieval recall across all dev set policies.
    retrieval_fn: callable that takes a query string
                  and returns list of retrieved chunks
    """
    recalls      = []
    all_missing  = []

    for entry in dev_set:
        query      = entry["query"]
        gt_actions = entry["actions"]

        retrieved = retrieval_fn(query)
        metrics   = measure_retrieval_recall(gt_actions, retrieved)

        recalls.append(metrics["retrieval_recall"])
        all_missing.extend(metrics["missing_from_retrieval"])

        if verbose:
            print(f"  {entry['policy_name']:<50} "
                  f"recall={metrics['retrieval_recall']:.3f}  "
                  f"missing={metrics['missing_from_retrieval'][:3]}")

    avg_recall   = sum(recalls) / len(recalls)
    perfect      = sum(1 for r in recalls if r == 1.0)
    zero         = sum(1 for r in recalls if r == 0.0)
    most_missed  = Counter(all_missing).most_common(10)

    print(f"\n{label}")
    print(f"  Avg Retrieval Recall:     {avg_recall:.3f}")
    print(f"  Recall = 1.0 (perfect):   {perfect}/{len(recalls)}")
    print(f"  Recall = 0.0 (nothing):   {zero}/{len(recalls)}")
    print(f"  Most frequently missed actions:")
    for action, count in most_missed[:5]:
        print(f"    {action:<45} missed in {count} policies")

    return recalls, avg_recall


print("✓ measure_retrieval_recall() defined")
print("✓ run_retrieval_evaluation() defined")

Dev set loaded: 100 policies
✓ measure_retrieval_recall() defined
✓ run_retrieval_evaluation() defined


In [ ]:
# ── Companion action rules — post-retrieval augmentation ─────────────
# These deterministic rules append known companion actions that are
# semantically distant from natural language queries but always required
# alongside certain primary actions.
# Applied after retrieval, before returning chunks to the generator.

COMPANION_RULES = {
    # ── EC2 resource creation ─────────────────────────────────────────
    "ec2:RunInstances":                  ["ec2:CreateTags",
                                          "ec2:DescribeInstances",
                                          "ec2:DescribeInstanceStatus",
                                          "ec2:DescribeImages",
                                          "ec2:DescribeSubnets",
                                          "ec2:DescribeVpcs",
                                          "ec2:DescribeSecurityGroups"],
    "ec2:CreateNetworkInterface":        ["ec2:CreateTags",
                                          "ec2:DescribeSubnets",
                                          "ec2:DescribeVpcs",
                                          "ec2:DescribeNetworkInterfaces",
                                          "ec2:DescribeSecurityGroups"],
    "ec2:CreateSecurityGroup":           ["ec2:CreateTags",
                                          "ec2:DescribeVpcs",
                                          "ec2:DescribeSecurityGroups"],
    "ec2:DescribeInstances":             ["ec2:DescribeInstanceStatus",
                                          "ec2:DescribeImages"],
    "ec2:StartInstances":                ["ec2:StopInstances",
                                          "ec2:DescribeInstances",
                                          "ec2:DescribeInstanceStatus"],

    # ── ECS task operations ───────────────────────────────────────────
    "ecs:RunTask":                       ["ec2:CreateTags",
                                          "ec2:DescribeNetworkInterfaces",
                                          "ec2:DescribeSubnets",
                                          "ec2:DescribeSecurityGroups",
                                          "iam:PassRole"],
    "ecs:CreateService":                 ["ec2:CreateTags",
                                          "ec2:DescribeSubnets",
                                          "ec2:DescribeVpcs",
                                          "ec2:DescribeSecurityGroups",
                                          "iam:PassRole"],
    "ecs:RegisterTaskDefinition":        ["iam:PassRole"],
    "ecs:UpdateService":                 ["iam:PassRole"],

    # ── IAM role passing ──────────────────────────────────────────────
    "codepipeline:StartPipelineExecution": ["iam:PassRole"],
    "codepipeline:CreatePipeline":         ["iam:PassRole"],
    "cloudformation:CreateStack":          ["iam:PassRole",
                                            "iam:CreateRole"],
    "cloudformation:ExecuteChangeSet":     ["iam:PassRole"],
    "cloudformation:CreateChangeSet":      ["iam:PassRole"],
    "codebuild:StartBuild":                ["iam:PassRole"],
    "glue:StartJobRun":                    ["iam:PassRole"],
    "lambda:CreateFunction":               ["iam:PassRole"],
    "states:StartExecution":               ["iam:PassRole"],

    # ── SSM commands need EC2 describe ───────────────────────────────
    "ssm:SendCommand":                   ["ec2:DescribeInstances",
                                          "ec2:DescribeInstanceStatus"],
    "ssm:StartAutomationExecution":      ["ec2:DescribeInstances",
                                          "ec2:DescribeInstanceStatus"],
    "ssm:GetCommandInvocation":          ["ssm:SendCommand",
                                          "ssm:ListCommands"],

    # ── CloudWatch Logs — always need all three ───────────────────────
    "logs:CreateLogStream":              ["logs:CreateLogGroup",
                                          "logs:PutLogEvents"],
    "logs:PutLogEvents":                 ["logs:CreateLogGroup",
                                          "logs:CreateLogStream"],
    "logs:CreateLogGroup":               ["logs:CreateLogStream",
                                          "logs:PutLogEvents"],

    # ── Kinesis stream reading ────────────────────────────────────────
    "kinesis:GetRecords":                ["kinesis:GetShardIterator",
                                          "kinesis:DescribeStream",
                                          "kinesis:ListStreams",
                                          "kinesis:DescribeStreamSummary"],
    "kinesis:GetShardIterator":          ["kinesis:GetRecords",
                                          "kinesis:DescribeStream",
                                          "kinesis:ListShards"],
    "kinesis:PutRecord":                 ["kinesis:PutRecords",
                                          "kinesis:DescribeStream"],

    # ── ECR image pulling ─────────────────────────────────────────────
    "ecr:BatchGetImage":                 ["ecr:GetAuthorizationToken",
                                          "ecr:BatchCheckLayerAvailability",
                                          "ecr:GetDownloadUrlForLayer"],
    "ecr:GetDownloadUrlForLayer":        ["ecr:GetAuthorizationToken",
                                          "ecr:BatchGetImage",
                                          "ecr:BatchCheckLayerAvailability"],
    "ecr:BatchCheckLayerAvailability":   ["ecr:GetAuthorizationToken",
                                          "ecr:BatchGetImage",
                                          "ecr:GetDownloadUrlForLayer"],

    # ── ECR image pushing ─────────────────────────────────────────────
    "ecr:PutImage":                      ["ecr:GetAuthorizationToken",
                                          "ecr:InitiateLayerUpload",
                                          "ecr:UploadLayerPart",
                                          "ecr:CompleteLayerUpload",
                                          "ecr:BatchCheckLayerAvailability"],
    "ecr:InitiateLayerUpload":           ["ecr:GetAuthorizationToken",
                                          "ecr:UploadLayerPart",
                                          "ecr:CompleteLayerUpload",
                                          "ecr:PutImage"],

    # ── DynamoDB streams ──────────────────────────────────────────────
    "dynamodb:GetRecords":               ["dynamodb:GetShardIterator",
                                          "dynamodb:DescribeStream",
                                          "dynamodb:ListStreams"],
    "dynamodb:GetShardIterator":         ["dynamodb:GetRecords",
                                          "dynamodb:DescribeStream",
                                          "dynamodb:ListStreams"],

    # ── SQS consumer pattern ──────────────────────────────────────────
    "sqs:ReceiveMessage":                ["sqs:DeleteMessage",
                                          "sqs:GetQueueAttributes"],
    "sqs:DeleteMessage":                 ["sqs:ReceiveMessage",
                                          "sqs:GetQueueAttributes"],

    # ── S3 bucket access ──────────────────────────────────────────────
    "s3:GetObject":                      ["s3:ListBucket"],
    "s3:PutObject":                      ["s3:GetObject",
                                          "s3:ListBucket"],
    "s3:ListBucket":                     ["s3:GetObject",
                                          "s3:GetBucketLocation"],

    # ── KMS encryption ────────────────────────────────────────────────
    "kms:Decrypt":                       ["kms:DescribeKey",
                                          "kms:GenerateDataKey"],
    "kms:GenerateDataKey":               ["kms:Decrypt",
                                          "kms:DescribeKey"],
    "kms:Encrypt":                       ["kms:DescribeKey",
                                          "kms:GenerateDataKey"],

    # ── Secrets Manager ───────────────────────────────────────────────
    "secretsmanager:GetSecretValue":     ["secretsmanager:DescribeSecret",
                                          "kms:Decrypt"],

    # ── CodeDeploy deployment ─────────────────────────────────────────
    "codedeploy:CreateDeployment":       ["codedeploy:GetDeployment",
                                          "codedeploy:GetDeploymentConfig",
                                          "codedeploy:GetDeploymentGroup"],
    "codedeploy:GetDeployment":          ["codedeploy:GetDeploymentConfig",
                                          "codedeploy:ListDeployments"],

    # ── Step Functions ────────────────────────────────────────────────
    "states:StartExecution":             ["states:DescribeExecution",
                                          "states:GetExecutionHistory"],
    "states:DescribeExecution":          ["states:GetExecutionHistory",
                                          "states:ListExecutions"],

    # ── RDS ───────────────────────────────────────────────────────────
    "rds:CreateDBInstance":              ["rds:DescribeDBInstances",
                                          "rds:DescribeDBSubnetGroups",
                                          "ec2:DescribeVpcs",
                                          "ec2:DescribeSubnets",
                                          "ec2:DescribeSecurityGroups"],
    "rds:ModifyDBInstance":              ["rds:DescribeDBInstances"],

    # ── Lambda event source mappings ──────────────────────────────────
    "lambda:CreateEventSourceMapping":   ["lambda:ListEventSourceMappings",
                                          "lambda:GetEventSourceMapping"],
    "lambda:InvokeFunction":             ["lambda:GetFunction"],

    # ── CloudWatch metrics ────────────────────────────────────────────
    "cloudwatch:PutMetricData":          ["cloudwatch:GetMetricStatistics",
                                          "cloudwatch:ListMetrics"],
    "cloudwatch:PutMetricAlarm":         ["cloudwatch:DescribeAlarms",
                                          "cloudwatch:GetMetricData"],
}


def apply_companion_rules(retrieved_chunks, dense_retriever):
    # Collect retrieved actions — normalized to lowercase
    retrieved_actions = set()
    for chunk in retrieved_chunks:
        if chunk["metadata"]["type"] == "iam_action":
            action = normalize_action_name(
                chunk["metadata"].get("action", "")
            ).lower()  # ← lowercase
            retrieved_actions.add(action)

    # Build lowercase version of COMPANION_RULES
    companion_rules_lower = {
        k.lower(): [v.lower() for v in vals]
        for k, vals in COMPANION_RULES.items()
    }

    # Find companions needed
    companions_to_add = set()
    for action in retrieved_actions:
        if action in companion_rules_lower:
            for companion in companion_rules_lower[action]:
                if companion not in retrieved_actions:
                    companions_to_add.add(companion)

    if not companions_to_add:
        return retrieved_chunks

    # Look up companion chunks from corpus — compare lowercase
    companion_chunks = []
    for chunk in dense_retriever.chunks:
        if chunk["metadata"]["type"] != "iam_action":
            continue
        action = normalize_action_name(
            chunk["metadata"].get("action", "")
        ).lower()  # ← lowercase
        if action in companions_to_add:
            companion_chunk = dict(chunk)
            companion_chunk["source"] = "companion_rule"
            companion_chunk["score"]  = 1.0
            companion_chunks.append(companion_chunk)
            companions_to_add.discard(action)

        if not companions_to_add:
            break

    return retrieved_chunks + companion_chunks


print("✓ COMPANION_RULES defined")
print(f"  {len(COMPANION_RULES)} trigger actions")
print("✓ apply_companion_rules() defined")

✓ COMPANION_RULES defined
  52 trigger actions
✓ apply_companion_rules() defined


In [ ]:
#test out the rrf+decomposer method
def retrieve_for_generation_decomposed_v3_hybrid(
    query,
    dense_retriever,
    sparse_retriever,
    openai_client,
    k_per_subquery=4,
    k_policies=2,
    score_threshold=0.55,
    dense_weight=0.7,
    sparse_weight=0.3
):
    """
    Decomposed retrieval with Dense+SPLADE hybrid per sub-query.
    Uses weighted RRF to combine dense and sparse rankings
    for each service-specific sub-query independently.
    """
    sub_queries = decompose_query_gpt4o(query, openai_client)

    print(f"  GPT-4o decomposed into {len(sub_queries)} sub-queries:")
    for sq in sub_queries:
        tag = f"[{sq['type']}]"
        print(f"    {tag:<12} {sq['service']}: {sq['query']}")

    seen_actions = set()
    all_actions  = []

    for sub_query_info in sub_queries:
        sub_query = sub_query_info['query']

        # Dense retrieval for this sub-query
        dense_results = dense_retriever.retrieve(sub_query, k=30)
        dense_actions = [
            r for r in dense_results
            if r['metadata']['type'] == 'iam_action'
        ]

        # SPLADE retrieval for this sub-query
        sparse_results = sparse_retriever.retrieve(sub_query, k=30)
        sparse_actions = [
            r for r in sparse_results
            if r['metadata']['type'] == 'iam_action'
        ]

        # Build action→chunk lookup from dense results
        # (SPLADE returns indices into the same chunk list)
        action_to_chunk = {}
        for r in dense_actions:
            action = normalize_action_name(
                r['metadata'].get('action', '')
            )
            if action not in action_to_chunk:
                action_to_chunk[action] = r

        # Build rank lookups for weighted RRF
        dense_ranks  = {
            normalize_action_name(
                r['metadata'].get('action', '')
            ): rank
            for rank, r in enumerate(dense_actions)
        }
        sparse_ranks = {
            normalize_action_name(
                r['metadata'].get('action', '')
            ): rank
            for rank, r in enumerate(sparse_actions)
        }

        # Weighted RRF across all actions seen in either retriever
        all_action_names = set(dense_ranks) | set(sparse_ranks)
        k_rrf = 60

        rrf_scores = {}
        for action in all_action_names:
            score = 0.0
            if action in dense_ranks:
                score += dense_weight * (
                    1.0 / (k_rrf + dense_ranks[action] + 1)
                )
            if action in sparse_ranks:
                score += sparse_weight * (
                    1.0 / (k_rrf + sparse_ranks[action] + 1)
                )
            rrf_scores[action] = score

        # Sort by combined RRF score
        ranked_actions = sorted(
            rrf_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        # Apply score threshold using dense score as reference
        # (RRF scores are not directly comparable to cosine similarity)
        # Use dense score for the threshold check
        dense_score_lookup = {
            normalize_action_name(
                r['metadata'].get('action', '')
            ): r['score']
            for r in dense_actions
        }

        added = 0
        for action, rrf_score in ranked_actions:
            clean = action.split(' [')[0]

            # Use dense score for threshold if available
            # otherwise use a relaxed threshold for sparse-only results
            ref_score = dense_score_lookup.get(action, score_threshold)
            if ref_score < score_threshold and action not in dense_score_lookup:
                continue

            if clean not in seen_actions and added < k_per_subquery:
                # Get the chunk — prefer dense chunk for metadata completeness
                chunk = action_to_chunk.get(action)
                if chunk is None:
                    # Only in sparse results — find it from sparse
                    for r in sparse_actions:
                        if normalize_action_name(
                            r['metadata'].get('action', '')
                        ) == action:
                            chunk = r
                            break

                if chunk is None:
                    continue

                result = dict(chunk)
                result['source']    = f"action_{sub_query_info['type']}_hybrid"
                result['sub_query'] = sub_query
                result['rrf_score'] = rrf_score

                seen_actions.add(clean)
                all_actions.append(result)
                added += 1

    # Second pass — policies using dense only (same as v3)
    seen_policies = set()
    all_policies  = []

    for sub_query_info in sub_queries:
        sub_query   = sub_query_info['query']
        results_sq  = dense_retriever.retrieve(sub_query, k=30)
        policies_sq = [
            r for r in results_sq
            if r['metadata']['type'] == 'iam_policy'
        ]

        for r in policies_sq:
            name     = r['metadata'].get('policy_name', '')
            n_actions = len(r['metadata'].get('actions_used', []))
            if name not in seen_policies and n_actions <= 15:
                seen_policies.add(name)
                r['source'] = 'policy_decomposed_hybrid'
                all_policies.append(r)
                break

    results = all_actions.copy()
    for r in all_policies[:k_policies]:
        results.append(r)

    print(f"  Total retrieved: {len(results)} chunks "
          f"({len(all_actions)} actions + "
          f"{len(results)-len(all_actions)} policies)")

    results = apply_companion_rules(results, dense_retriever)
    return results


print("✓ retrieve_for_generation_decomposed_v3_hybrid() defined")

✓ retrieve_for_generation_decomposed_v3_hybrid() defined


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# k parameter sweep
# Each retrieval strategy optimized independently
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("K-PARAMETER SWEEP")
print("=" * 65)

# ── Part A: Dense — sweep k_actions (fix k_policies=2) ───────────────
print("\nPart A — Dense: sweep k_actions (k_policies fixed at 2):")
print(f"  {'k_actions':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_actions_results = {}
for k_actions in [2, 4, 6, 8, 10]:
    def fn(query, k=k_actions):
        return retrieve_for_generation(
            query, dense_retriever,
            k_actions=k, k_policies=2
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_actions_results[k_actions] = avg
    print(f"  k_actions={k_actions:<5} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_actions = max(k_actions_results, key=k_actions_results.get)
print(f"\n  → Best k_actions: {best_k_actions} "
      f"(recall={k_actions_results[best_k_actions]:.3f})")


# ── Part B: Dense — sweep k_policies (fix k_actions at best) ─────────
print(f"\nPart B — Dense: sweep k_policies "
      f"(k_actions fixed at {best_k_actions}):")
print(f"  {'k_policies':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_policies_results = {}
for k_policies in [1, 2, 3, 4]:
    def fn(query, k=k_policies):
        return retrieve_for_generation(
            query, dense_retriever,
            k_actions=best_k_actions, k_policies=k
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_policies_results[k_policies] = avg
    print(f"  k_policies={k_policies:<4} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_policies_dense = max(k_policies_results, key=k_policies_results.get)
print(f"\n  → Best k_policies (Dense): {best_k_policies_dense} "
      f"(recall={k_policies_results[best_k_policies_dense]:.3f})")


# ── Part C: Decomposed v2 — sweep k_per_subquery independently ────────
print(f"\nPart C — Decomposed v2 (GPT-3.5): sweep k_per_subquery "
      f"(k_policies fixed at 2):")
print(f"  {'k_per_sq':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_per_sq_v2_results = {}
for k_per_sq in [1, 2, 3, 4]:
    def fn(query, k=k_per_sq):
        return retrieve_for_generation_decomposed_v2(
            query, dense_retriever, openai_client,
            k_per_subquery=k, k_policies=2
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_per_sq_v2_results[k_per_sq] = avg
    print(f"  k_per_sq={k_per_sq:<5} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_per_sq_v2 = max(k_per_sq_v2_results, key=k_per_sq_v2_results.get)
print(f"\n  → Best k_per_subquery (v2): {best_k_per_sq_v2} "
      f"(recall={k_per_sq_v2_results[best_k_per_sq_v2]:.3f})")


# ── Part D: Decomposed v2 — sweep k_policies (fix k_per_sq at best) ──
print(f"\nPart D — Decomposed v2: sweep k_policies "
      f"(k_per_subquery fixed at {best_k_per_sq_v2}):")
print(f"  {'k_policies':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_policies_v2_results = {}
for k_policies in [1, 2, 3, 4]:
    def fn(query, k=k_policies):
        return retrieve_for_generation_decomposed_v2(
            query, dense_retriever, openai_client,
            k_per_subquery=best_k_per_sq_v2, k_policies=k
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_policies_v2_results[k_policies] = avg
    print(f"  k_policies={k_policies:<4} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_policies_v2 = max(k_policies_v2_results, key=k_policies_v2_results.get)
print(f"\n  → Best k_policies (v2): {best_k_policies_v2} "
      f"(recall={k_policies_v2_results[best_k_policies_v2]:.3f})")


# ── Part E: Decomposed v3 — sweep k_per_subquery independently ────────
print(f"\nPart E — Decomposed v3 (GPT-4o): sweep k_per_subquery "
      f"(k_policies fixed at 2):")
print(f"  {'k_per_sq':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_per_sq_v3_results = {}
for k_per_sq in [1, 2, 3, 4]:
    def fn(query, k=k_per_sq):
        return retrieve_for_generation_decomposed_v3(
            query, dense_retriever, openai_client,
            k_per_subquery=k, k_policies=2
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_per_sq_v3_results[k_per_sq] = avg
    print(f"  k_per_sq={k_per_sq:<5} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_per_sq_v3 = max(k_per_sq_v3_results, key=k_per_sq_v3_results.get)
print(f"\n  → Best k_per_subquery (v3): {best_k_per_sq_v3} "
      f"(recall={k_per_sq_v3_results[best_k_per_sq_v3]:.3f})")


# ── Part F: Decomposed v3 — sweep k_policies (fix k_per_sq at best) ──
print(f"\nPart F — Decomposed v3: sweep k_policies "
      f"(k_per_subquery fixed at {best_k_per_sq_v3}):")
print(f"  {'k_policies':<12} {'Avg Recall':<14} {'Perfect':>10}")
print("  " + "─" * 40)

k_policies_v3_results = {}
for k_policies in [1, 2, 3, 4]:
    def fn(query, k=k_policies):
        return retrieve_for_generation_decomposed_v3(
            query, dense_retriever, openai_client,
            k_per_subquery=best_k_per_sq_v3, k_policies=k
        )
    recalls, avg = run_retrieval_evaluation(dev_set, fn, label="")
    perfect = sum(1 for r in recalls if r == 1.0)
    k_policies_v3_results[k_policies] = avg
    print(f"  k_policies={k_policies:<4} {avg:<14.3f} {perfect}/{len(dev_set)}")

best_k_policies_v3 = max(k_policies_v3_results, key=k_policies_v3_results.get)
print(f"\n  → Best k_policies (v3): {best_k_policies_v3} "
      f"(recall={k_policies_v3_results[best_k_policies_v3]:.3f})")


# ── Summary of best k values found ───────────────────────────────────
print(f"\n{'='*65}")
print(f"K-SWEEP CONCLUSIONS")
print(f"{'='*65}")
print(f"  Dense:          k_actions={best_k_actions}, "
      f"k_policies={best_k_policies_dense}")
print(f"  Decomposed v2:  k_per_subquery={best_k_per_sq_v2}, "
      f"k_policies={best_k_policies_v2}")
print(f"  Decomposed v3:  k_per_subquery={best_k_per_sq_v3}, "
      f"k_policies={best_k_policies_v3}")

Streaming output truncated to the last 5000 lines.
    [explicit]   ssm: SSM describe patch baselines list patch baselines get patch baseline
               reason: needs describe, list, get actions to retrieve and list patch
    [explicit]   tag: Tagging get resources list tags for resource
               reason: needs get and list actions to access resource tagging inform
  Total retrieved: 3 chunks (2 actions + 1 policies)
  GPT-4o decomposed into 2 sub-queries:
    [explicit]   ec2: EC2 describe instances modify instance attribute
               reason: needs describe instances, modify instance attribute for upda
    [explicit]   ssm: SSM send command describe instance information
               reason: needs send command, describe instance information for managi
  Total retrieved: 3 chunks (2 actions + 1 policies)
  GPT-4o decomposed into 4 sub-queries:
    [explicit]   ecr: ECR batch get image put image list images
               reason: needs to upload and manage image layers, s

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Retrieval strategy ablation on dev set
# Each strategy compared at its own best k values
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("RETRIEVAL STRATEGY ABLATION")
print("Each strategy evaluated at its own best k values")
print("=" * 65)

best_k_actions       = 8
best_k_policies_dense = 3
best_k_per_sq_v2     = 4
best_k_policies_v2   = 4
best_k_per_sq_v3     = 4
best_k_policies_v3   = 2

strategies = {
    f"Dense (k_act={best_k_actions}, k_pol={best_k_policies_dense})":
        lambda q: retrieve_for_generation(
            q, dense_retriever,
            k_actions=best_k_actions,
            k_policies=best_k_policies_dense
        ),
    f"RRF Hybrid (k_act={best_k_actions}, k_pol={best_k_policies_dense})":
        lambda q: retrieve_for_generation_rrf(
            q, dense_retriever, bm25_retriever,
            k_actions=best_k_actions,
            k_policies=best_k_policies_dense
        ),
    f"Decomposed v2 (k_sq={best_k_per_sq_v2}, k_pol={best_k_policies_v2})":
        lambda q: retrieve_for_generation_decomposed_v2(
            q, dense_retriever, openai_client,
            k_per_subquery=best_k_per_sq_v2,
            k_policies=best_k_policies_v2
        ),
    f"Decomposed v3 (k_sq={best_k_per_sq_v3}, k_pol={best_k_policies_v3})":
        lambda q: retrieve_for_generation_decomposed_v3(
            q, dense_retriever, openai_client,
            k_per_subquery=best_k_per_sq_v3,
            k_policies=best_k_policies_v3
        ),
    f"Decomposed v3 Hybrid (k_sq={best_k_per_sq_v3}, k_pol={best_k_policies_v3})":
        lambda q: retrieve_for_generation_decomposed_v3_hybrid(
            q, dense_retriever, sparse_retriever, openai_client,
            k_per_subquery=best_k_per_sq_v3,
            k_policies=best_k_policies_v3,
            dense_weight=0.7,
            sparse_weight=0.3
        )
}


def run_retrieval_evaluation_extended(dev_set, retrieval_fn, label,
                                       hit_k_values=(1, 3, 5),
                                       verbose=False):
    """
    Extended retrieval evaluation measuring:
    - Action-only recall (fraction of GT actions surfaced)
    - Action-only precision (fraction of retrieved actions in GT)
    - Hit@K (fraction of policies where at least 1 GT action retrieved
             in top-K retrieved action chunks)
    - Most frequently missed actions
    """
    recalls           = []
    precisions        = []
    all_missing       = []
    hit_counts        = {k: 0 for k in hit_k_values}
    n_retrieved_total = []

    for entry in dev_set:
        query      = entry["query"]
        gt_actions = {
            normalize_action_name(a).lower()
            for a in entry["actions"]
        }

        retrieved = retrieval_fn(query)

        # All retrieved action chunks in order
        retrieved_action_chunks = [
            r for r in retrieved
            if r["metadata"]["type"] == "iam_action"
        ]

        # Full retrieved action set (for recall and precision)
        retrieved_action_names = {
            normalize_action_name(
                r["metadata"].get("action", "")
            ).lower()
            for r in retrieved_action_chunks
        }

        # ── Recall ───────────────────────────────────────────────────
        covered = gt_actions & retrieved_action_names
        recall  = len(covered) / len(gt_actions) if gt_actions else 0
        recalls.append(recall)
        all_missing.extend(gt_actions - retrieved_action_names)

        # ── Precision ────────────────────────────────────────────────
        # Fraction of retrieved action chunks that are in GT
        precision = (
            len(covered) / len(retrieved_action_names)
            if retrieved_action_names else 0
        )
        precisions.append(precision)
        n_retrieved_total.append(len(retrieved_action_chunks))

        # ── Hit@K ────────────────────────────────────────────────────
        # Check if at least 1 GT action appears in top-K retrieved chunks
        for k in hit_k_values:
            top_k_actions = {
                normalize_action_name(
                    r["metadata"].get("action", "")
                ).lower()
                for r in retrieved_action_chunks[:k]
            }
            if top_k_actions & gt_actions:
                hit_counts[k] += 1

        if verbose:
            print(f"  {entry['policy_name']:<50} "
                  f"recall={recall:.3f} "
                  f"precision={precision:.3f}")

    n = len(dev_set)
    avg_recall    = sum(recalls)    / n
    avg_precision = sum(precisions) / n
    f1            = (
        2 * avg_precision * avg_recall /
        (avg_precision + avg_recall)
        if (avg_precision + avg_recall) > 0 else 0
    )
    avg_retrieved = sum(n_retrieved_total) / n
    perfect       = sum(1 for r in recalls    if r == 1.0)
    zero          = sum(1 for r in recalls    if r == 0.0)
    hit_rates     = {k: hit_counts[k] / n for k in hit_k_values}

    most_missed = Counter(all_missing).most_common(5)

    if label:
        print(f"\n{label}")
        print(f"  Avg Recall:              {avg_recall:.3f}")
        print(f"  Avg Precision:           {avg_precision:.3f}")
        print(f"  Avg F1:                  {f1:.3f}")
        print(f"  Avg Retrieved Actions:   {avg_retrieved:.1f}")
        print(f"  Recall = 1.0 (perfect):  {perfect}/{n}")
        print(f"  Recall = 0.0 (nothing):  {zero}/{n}")
        for k in hit_k_values:
            print(f"  Hit@{k}:                  "
                  f"{hit_rates[k]:.3f} ({hit_counts[k]}/{n})")
        print(f"  Most frequently missed actions:")
        for action, count in most_missed:
            print(f"    {action:<45} missed in {count} policies")

    return {
        "recalls":       recalls,
        "precisions":    precisions,
        "avg_recall":    avg_recall,
        "avg_precision": avg_precision,
        "avg_f1":        f1,
        "avg_retrieved": avg_retrieved,
        "perfect":       perfect,
        "zero":          zero,
        "hit_rates":     hit_rates,
        "most_missed":   most_missed,
    }


# ── Run extended ablation ─────────────────────────────────────────────
ablation_results = {}
for label, fn in strategies.items():
    res = run_retrieval_evaluation_extended(
        dev_set, fn, label=label, hit_k_values=(1, 3, 5)
    )
    ablation_results[label] = res


# ── Summary table ─────────────────────────────────────────────────────
print("\n\n" + "=" * 95)
print("ABLATION SUMMARY")
print("=" * 95)
print(f"  {'Strategy':<45} {'Recall':>8} {'Precis':>8} {'F1':>8} "
      f"{'Hit@1':>7} {'Hit@3':>7} {'Hit@5':>7} "
      f"{'Perfect':>8} {'Zero':>6}")
print("  " + "─" * 95)

for label, res in sorted(
    ablation_results.items(),
    key=lambda x: x[1]["avg_recall"],
    reverse=True
):
    n = len(dev_set)
    print(
        f"  {label:<45} "
        f"{res['avg_recall']:>8.3f} "
        f"{res['avg_precision']:>8.3f} "
        f"{res['avg_f1']:>8.3f} "
        f"{res['hit_rates'][1]:>7.3f} "
        f"{res['hit_rates'][3]:>7.3f} "
        f"{res['hit_rates'][5]:>7.3f} "
        f"{res['perfect']:>6}/{n} "
        f"{res['zero']:>4}/{n}"
    )

best_strategy = max(
    ablation_results,
    key=lambda x: ablation_results[x]["avg_recall"]
)
print(f"\n  → Best strategy by recall:    {best_strategy}")
print(f"     Avg recall:    "
      f"{ablation_results[best_strategy]['avg_recall']:.3f}")
print(f"     Avg precision: "
      f"{ablation_results[best_strategy]['avg_precision']:.3f}")
print(f"     Avg F1:        "
      f"{ablation_results[best_strategy]['avg_f1']:.3f}")


RETRIEVAL STRATEGY ABLATION
Each strategy evaluated at its own best k values

Dense (k_act=8, k_pol=3)
  Avg Recall:              0.538
  Avg Precision:           0.341
  Avg F1:                  0.417
  Avg Retrieved Actions:   9.2
  Recall = 1.0 (perfect):  18/100
  Recall = 0.0 (nothing):  11/100
  Hit@1:                  0.620 (62/100)
  Hit@3:                  0.820 (82/100)
  Hit@5:                  0.880 (88/100)
  Most frequently missed actions:
    ec2:describevpcs                              missed in 8 policies
    cloudwatch:getmetricdata                      missed in 8 policies
    iam:passrole                                  missed in 8 policies
    cloudwatch:describealarms                     missed in 7 policies
    ec2:createtags                                missed in 7 policies

RRF Hybrid (k_act=8, k_pol=3)
  Avg Recall:              0.353
  Avg Precision:           0.228
  Avg F1:                  0.277
  Avg Retrieved Actions:   9.0
  Recall = 1.0 (perfect):

In [ ]:
# ── Generation functions ─────────────────────────

import openai
from google.colab import userdata

openai_client = openai.OpenAI(
    api_key=userdata.get('OPENAI_API_KEY')
)


def generate_policy_no_rag(query):
    """Generate IAM policy with NO retrieval context."""
    prompt = f"""You are an AWS IAM expert.
Generate a valid IAM policy JSON document for this task:

"{query}"

Requirements:
- Return ONLY valid JSON, no explanation
- Follow exact IAM policy JSON structure
- Use only real AWS IAM action names
- Include Version and Statement fields

IAM Policy:"""

    response = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()


def generate_policy_with_rag(query, retrieved_chunks):
    """Generate IAM policy WITH retrieved context."""

    # ── Pass 1: services directly named in retrieved action chunks ──────
    direct_services = {
        r['metadata']['action'].split(':')[0]
        for r in retrieved_chunks
        if r['metadata']['type'] == 'iam_action'
    }

    # ── Pass 1 policy scan: collect ALL actions from relevant policies ──
    # We need a first pass to discover dependent services (e.g. a policy
    # for an ECS task also contains ecr:* and logs:* actions — those
    # service prefixes must be allowed through in the final filter).
    dependent_services = set()
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_policy':
            continue
        doc = r['metadata']['policy_document']
        for stmt in doc.get('Statement', []):
            acts = stmt.get('Action', [])
            if isinstance(acts, str):
                acts = [acts]
            for a in acts:
                svc = a.split(':')[0]
                # Only expand from actions whose service is already relevant
                # — prevents completely unrelated services bleeding in
                if svc in direct_services or '*' in a:
                    dependent_services.add(svc)
                    # Also capture the co-occurring services in this stmt
                    for a2 in (acts if isinstance(acts, list) else [acts]):
                        dependent_services.add(a2.split(':')[0])

    relevant_services = direct_services | dependent_services

    # ── Action context: name + description + resource types ────────────
    action_lines = []
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_action':
            continue
        action         = r['metadata']['action']
        description    = r['metadata'].get('description', '')[:80]
        resource_types = r['metadata'].get('resource_types', [])
        resource_str   = (
            '  [resources: ' + ', '.join(resource_types) + ']'
            if resource_types else ''
        )
        action_lines.append('- ' + action + ': ' + description + resource_str)

    action_context = '\n'.join(action_lines)

    # ── Policy context: filtered to relevant + dependent services ───────
    policy_sections = []
    for r in retrieved_chunks:
        if r['metadata']['type'] != 'iam_policy':
            continue
        doc = r['metadata']['policy_document']

        filtered_actions = []
        for stmt in doc.get('Statement', []):
            acts = stmt.get('Action', [])
            if isinstance(acts, str):
                acts = [acts]
            for a in acts:
                service = a.split(':')[0]
                if (not relevant_services
                        or '*' in a
                        or service in relevant_services):
                    filtered_actions.append(a)

        if not filtered_actions:
            continue

        section = (
            'Policy name: ' + r['metadata']['policy_name'] + '\n'
            'Relevant actions from this policy:\n'
            + '\n'.join('  - ' + a for a in filtered_actions)
        )
        policy_sections.append(section)

    policy_context = '\n\n'.join(policy_sections)

    # ── Build prompt ────────────────────────────────────────────────────
    prompt = (
        'You are an AWS IAM expert.\n'
        'Generate a valid IAM policy JSON for this task:\n\n'
        '"' + query + '"\n\n'
        'INSTRUCTIONS — follow these steps in order:\n\n'
        'STEP 1: Read the example policies below carefully.\n'
        'The actions listed are REAL verified AWS actions.\n'
        'Include ALL actions from these policies that are\n'
        'relevant to the task above.\n\n'
        '<verified_policies>\n'
        + policy_context +
        '\n</verified_policies>\n\n'
        'STEP 2: The following verified actions are also relevant\n'
        'to this task — include all of them:\n\n'
        '<verified_actions>\n'
        + action_context +
        '\n</verified_actions>\n\n'
        'STEP 3: Add any other real AWS actions you know are required\n'
        'for this task that are not already included above.\n'
        'Pay special attention to implicit dependencies:\n'
        '  - Auth tokens required before accessing a service\n'
        '  - KMS decrypt when reading encrypted resources\n'
        '  - Logging actions (logs:CreateLogGroup, logs:CreateLogStream,\n'
        '    logs:PutLogEvents) when a service writes execution logs\n\n'
        'IMPORTANT:\n'
        '- Include ALL permissions needed for the task to work\n'
        '- Do not omit actions just because they seem implicit\n'
        '- Return ONLY valid JSON with Version and Statement fields\n\n'
        'IAM Policy:'
    )

    try:
        import json as json_test
        json_test.dumps({'role': 'user', 'content': prompt})
    except Exception as e:
        print(f'Prompt serialization error: {e}')
        prompt = prompt.encode('utf-8', errors='ignore').decode('utf-8')

    response = openai_client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=2000
    )
    return response.choices[0].message.content.strip()


print('✓ generate_policy_with_rag() updated — expanded service filtering')

✓ generate_policy_with_rag() updated — expanded service filtering


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Retrieval recall vs generation recall gap analysis
# Shows how much the generator "loses" from good retrieved context
# Only runs on best strategy to keep API costs manageable
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("RETRIEVAL → GENERATION GAP ANALYSIS")
print(f"Strategy: {best_strategy}")
print("=" * 65)
print("Note: runs generation — will make GPT-3.5 API calls")
print()

# Use a subset of dev set for gap analysis to control API cost
import random
random.seed(42)
gap_sample = random.sample(dev_set, min(30, len(dev_set)))
print(f"Running on {len(gap_sample)} sampled dev policies...")

retrieval_recalls  = []
generation_recalls = []
gaps               = []

best_fn = strategies[best_strategy]

for entry in gap_sample:
    query      = entry["query"]
    gt_actions = entry["actions"]

    # Measure retrieval recall
    retrieved   = best_fn(query)
    ret_metrics = measure_retrieval_recall(gt_actions, retrieved)

    # Measure generation recall
    raw_output = generate_policy_with_rag(query, retrieved)
    gen_actions, policy = extract_actions(raw_output)

    if policy is None:
        continue

    gt_set     = {normalize_action_name(a).lower()
                  for a in gt_actions}
    gen_set    = {normalize_action_name(a).lower()
                  for a in gen_actions}
    gen_recall = len(gt_set & gen_set) / len(gt_set) if gt_set else 0

    ret_recall = ret_metrics["retrieval_recall"]
    gap        = ret_recall - gen_recall

    retrieval_recalls.append(ret_recall)
    generation_recalls.append(gen_recall)
    gaps.append(gap)

    if gap > 0.2:  # flag large gaps for inspection
        print(f"  LARGE GAP: {entry['policy_name']}")
        print(f"    Retrieval recall:  {ret_recall:.3f}")
        print(f"    Generation recall: {gen_recall:.3f}")
        print(f"    Gap:               {gap:.3f}")
        print(f"    Missing from ret:  "
              f"{ret_metrics['missing_from_retrieval']}")
        missing_in_gen = gt_set - gen_set
        in_retrieved   = missing_in_gen & {
            normalize_action_name(
                c["metadata"].get("action", "")
            ).lower()
            for c in retrieved
            if c["metadata"]["type"] == "iam_action"
        }
        print(f"    In retrieval but not generated: "
              f"{sorted(in_retrieved)}")

n = len(gaps)
avg_ret = sum(retrieval_recalls) / n if n else 0
avg_gen = sum(generation_recalls) / n if n else 0
avg_gap = sum(gaps) / n if n else 0

print(f"\nGAP ANALYSIS RESULTS (n={n}):")
print(f"  Avg Retrieval Recall:  {avg_ret:.3f}")
print(f"  Avg Generation Recall: {avg_gen:.3f}")
print(f"  Avg Gap:               {avg_gap:.3f}")
print()

if avg_gap < 0.05:
    diagnosis = "Generator faithfully uses retrieved context"
elif avg_gap < 0.15:
    diagnosis = "Small generator loss — minor prompt tuning may help"
else:
    diagnosis = "Large generator loss — generator ignoring context"

print(f"  Diagnosis: {diagnosis}")
print()
print(f"  Breakdown:")
small_gap  = sum(1 for g in gaps if g <= 0.05)
medium_gap = sum(1 for g in gaps if 0.05 < g <= 0.2)
large_gap  = sum(1 for g in gaps if g > 0.2)
print(f"    Gap ≤ 0.05 (generator uses context well): "
      f"{small_gap}/{n}")
print(f"    Gap 0.05–0.20 (some loss):                "
      f"{medium_gap}/{n}")
print(f"    Gap > 0.20 (generator ignoring context):  "
      f"{large_gap}/{n}")


RETRIEVAL → GENERATION GAP ANALYSIS
Strategy: Decomposed v3 (k_sq=4, k_pol=2)
Note: runs generation — will make GPT-3.5 API calls

Running on 30 sampled dev policies...
  GPT-4o decomposed into 1 sub-queries:
    [explicit]   iam: IAM list service specific credentials create service specific credential update service specific credential delete service specific credential reset service specific credential
               reason: needs list, create, update, delete, reset actions for managi
  Total retrieved: 5 chunks (4 actions + 1 policies)
  GPT-4o decomposed into 2 sub-queries:
    [explicit]   rds: RDS describe db clusters describe db instances list tags for resource
               reason: needs describe db clusters, describe db instances, list tags
    [explicit]   cloudwatch: CloudWatch get metric data list metrics
               reason: needs get metric data, list metrics to retrieve associated m
  Total retrieved: 10 chunks (8 actions + 2 policies)
  GPT-4o decomposed into 1 sub-